# 📌 Dynamic Vertical Agent Creation

![Topic](https://img.shields.io/badge/Topic-DVAC-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Agentic-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-April%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — Dynamic Vertical Agent Creation (that we will call here DVAC for simplicity) is the discipline of building self-organizing AI agent pipelines where a master agent autonomously decomposes tasks, spawns sub-agents, verifies reasoning outputs, and iterates.</span>

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

Compared to traditional AI automation that builds a single chain like input → model → output, DVAC constructs a **living system of cooperating agents** that can plan, delegate, self-check, and retry. 
<br>
<br>
**In brief**, a Master Agent understands the task, defines a **vertical** (healthcare, legal, finance, logistics), defines **prompt/tools/database** to use, creates **scripts**, analyzes the **result**. 

The **key architectural pillars** are:

1. **The Master Agent** that sits at the top of the hierarchy. It receives the high-level goal, breaks it into sub-tasks, routes them to specialized sub-agents, aggregates results, and decides whether to accept, retry, or escalate. It can be seen as the "brain" of the system.

2. **Multi-Agent Systems (MAS)** that are the backbone. Instead of one general model doing everything, you create here specialists that each operates in a specific lane, communicating through shared memory or message passing. A medical researcher agent, a writer agent, a validator agent or a tool-calling agent are examples of specialists. 

3. **Verifiable Reasoning** is the quality gate. While each agent produces evidence (chain-of-thought traces, confidence scores, structured proofs,ect), you need that specific agents (or the master agent itself) to verify these outputs before proceeding.

Recently, the flow had evolved from a chain (linear A→B→C) to a graph (non-linear, with branching, merging, and conditional routing). Within this graph, recursive loops allow agents to self-improve by sending feedback from an output that fails validation.
<br>
<br>
**Warning**: While Master Agent can build by itself tools, it still must be connected to **"seed tools"** (e.g., File system Access, Python Interpreter, Package Manager Access, ect) at first to be fully operational.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

### **2.1 Goal ingestion**

**The master agent** receives a high-level vertical-specific goal (e.g. "generate a compliance report for this medical trial") and decompose it into steps. 

### **2.2  From Chain to Graph**

Before, master agent chained the steps: Agent A → Agent B → Agent C (**Chain**). Now, the master agent dynamically decides which agents run in parallel, which run conditionally, and which depend on others (**Graph**).

### **2.3 Sub-agent execution (MAS)**

Specialized agents run their tasks. Each one produces **an answer AND a reasoned trace (structured evidence)** to make its output verifiable.

### **2.4  Verifiable Reasoning checkpoint**

A validator agent (or the master agent itself) **checks outputs**: Are citations present? Does the logic hold? Does confidence meet the threshold? If not, the output is rejected.

### **2.5  Recursive loop**

Failed outputs **re-enter the relevant sub-agent with feedback**. This loop can iterate several times before the master agent escalates or terminates with an error.

### **2.6  Output delivery**

A verified, structured and traceable output is **delivered** to the end system.

![DVAC](../assets/DVAC.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Vertical specialization** | Higher accuracy than general-purpose agents due to domain-specific training and context. |
| 🟢 | **Graph structure** | Handles complex workflows that sequential chains cannot manage. |
| 🟢 | **Verifiable reasoning** | Creates auditability with evidence trails. |
| 🟢 | **Recursive loops** | Catch and correct mistakes that a single-pass system would ship to production. |
| 🟢 | **Master agent architecture** | Scales by adding new sub-agents without rebuilding the entire system from scratch. |
| 🔴 | **Recursive loop risk** | Can spiral into infinite retries if exit conditions are poorly defined, creating runaway costs. |
| 🔴 | **Graph complexity** | Debugging a 20-agent graph is non-trivial and requires dedicated observability tooling. |
| 🔴 | **Verification latency** | Every verifiable reasoning step adds time and token cost making DVAC slower and more expensive than a single pass. |
| 🔴 | **Security surface**| Risks multiply with every agent boundary (e.g., prompt injection, privilege escalation threats,..). |
| 🔴 | **Vertical lock-in** | A healthcare system cannot simply be repurposed for logistics. |

---
## 4. Code Example

> **Goal:** Build a minimal DVAC skeleton: A master agent that routes a task through a graph of sub-agents (Researcher → Reasoner), validates outputs with verifiable reasoning, and retries via a recursive loop with a security gate.

In [ ]:
# ---------------------------------------------------------------------------
# Simple DVAC skeleton
# Demonstrates: master agent, MAS sub-agents, verifiable reasoning,
# chain→graph routing, recursive retry loop, and a basic security guard.
# ---------------------------------------------------------------------------

import json
from typing import Callable

MAX_RETRIES = 3  # ← guards against infinite recursive loops

# ---------------------------------------------------------------------------
# Security guard (prompt injection filter)
# ---------------------------------------------------------------------------

BANNED_PATTERNS = ["ignore previous", "you are now", "system:"]

def security_check(text: str) -> bool:
    """Returns True if the text is safe to process."""
    lower = text.lower()
    for pattern in BANNED_PATTERNS:
        if pattern in lower:
            print(f"[SECURITY] Blocked pattern detected: '{pattern}'")
            return False
    return True

# ---------------------------------------------------------------------------
# Sub-agents: Researcher, Reasoner, Validator
# ---------------------------------------------------------------------------

def researcher_agent(task: str) -> dict:
    """Retrieves domain knowledge. Returns answer + evidence trace."""
    # In production: RAG, vector DB, or web search
    return {
        "result": f"Domain data for: {task}",
        "evidence": ["source_1.pdf", "source_2.db"],
        "confidence": 0.85
    }

def reasoner_agent(task: str, context: dict) -> dict:
    """Builds structured reasoning on top of retrieved data."""
    return {
        "result": f"Reasoned conclusion for: {task} using {context['result']}",
        "chain_of_thought": ["Step 1: ...", "Step 2: ...", "Step 3: ..."],
        "confidence": 0.90
    }

def validator_agent(output: dict) -> tuple[bool, str]:
    """Verifiable Reasoning: checks confidence + evidence presence."""
    if output.get("confidence", 0) < 0.80:
        return False, "Confidence below threshold"
    if not output.get("evidence") and not output.get("chain_of_thought"):
        return False, "No verifiable trace found"
    return True, "OK"

# ---------------------------------------------------------------------------
# Graph router
# This replaces a fixed chain. The master agent picks which agents run
# and in what order — this is the chain→graph switch.
# ---------------------------------------------------------------------------

def route_task(task_type: str) -> list[Callable]:
    graph = {
        "research_heavy": [researcher_agent, reasoner_agent],
        "logic_heavy":    [reasoner_agent],
        "full_pipeline":  [researcher_agent, reasoner_agent],
    }
    return graph.get(task_type, [researcher_agent, reasoner_agent])

# ---------------------------------------------------------------------------
# Master agent
# ---------------------------------------------------------------------------

def master_agent(goal: str, task_type: str = "full_pipeline") -> dict:
    """
    Orchestrates the MAS pipeline with recursive retry on validation failure.
    Security check runs first — always.
    """

    # ── Security gate ──
    if not security_check(goal):
        return {"status": "BLOCKED", "reason": "Security policy violation"}

    agents = route_task(task_type)  # graph-based routing
    context = {}
    attempt = 0

    # ── Recursive loop ──
    while attempt < MAX_RETRIES:
        attempt += 1
        print(f"\n[Master] Attempt {attempt}/{MAX_RETRIES} — routing to {len(agents)} agents")

        # Run each sub-agent in sequence (could be parallel with asyncio)
        for agent_fn in agents:
            if agent_fn == researcher_agent:
                context = agent_fn(goal)
            else:
                context = agent_fn(goal, context)

        # Verifiable Reasoning checkpoint
        is_valid, message = validator_agent(context)

        if is_valid:
            print(f"[Master] Validation passed: {message}")
            return {"status": "SUCCESS", "output": context, "attempts": attempt}
        else:
            print(f"[Master] Validation failed: {message} — retrying...")
            context["confidence"] = context.get("confidence", 0) + 0.05  # simulate improvement

    return {"status": "FAILED", "reason": "Max retries exceeded", "last_output": context}

# ---------------------------------------------------------------------------
# Run it
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    goal = "Summarize patient trial outcomes for compound X in oncology vertical"
    result = master_agent(goal, task_type="full_pipeline")
    print("\n── Final Output ──")
    print(json.dumps(result, indent=2))

---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **Master agent** is a **coordinator**, not a worker.
- **Graph** is more powerful than the **chain** — but harder to debug.
- **Verifiable Reasoning** is what separates production-grade from toy systems.
- **Recursive loops** need hard exit conditions or they become your worst enemy.
- **Security risks** are exponential with the number of agents in MAS.

</div>